# Legal Document Type Classifier
**Model:** `nlpaueb/legal-bert-base-uncased`  
**Task:** Classify legal documents into 7 types (scotus, ledgar, eurlex, ecthr_a, ecthr_b, case_hold, unfair_tos)  
**Data:** 35k train | 7k val | 7k test — balanced, 5000 per class  

### Before running:
1. Runtime > Change runtime type > T4 GPU
2. Upload: `combined_legal_train.csv`, `combined_legal_validation.csv`, `combined_legal_test.csv`
3. Run all cells in order

In [ ]:
# Cell 1: Install
!pip install transformers scikit-learn -q

In [ ]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import torch
import os
import json
import shutil
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3: Load data
train_df = pd.read_csv('combined_legal_train.csv')
val_df   = pd.read_csv('combined_legal_validation.csv')
test_df  = pd.read_csv('combined_legal_test.csv')

print('Train shape:', train_df.shape)
print('Val shape:  ', val_df.shape)
print('Test shape: ', test_df.shape)
print('\nLabel distribution (train):')
print(train_df['label'].value_counts())
print('\nSample row:')
print(train_df.iloc[0])

In [ ]:
# Cell 4: Config
MODEL_NAME  = 'nlpaueb/legal-bert-base-uncased'
MAX_LENGTH  = 512
BATCH_SIZE  = 8
EPOCHS      = 3
LR          = 2e-5
OUTPUT_DIR  = './legal_doc_classifier_output'
SAVE_DIR    = './legal_doc_classifier_final'

TEXT_COL  = 'text'
LABEL_COL = 'label'

In [ ]:
# Cell 5: Encode labels
le = LabelEncoder()
le.fit(train_df[LABEL_COL])

train_labels = le.transform(train_df[LABEL_COL])
val_labels   = le.transform(val_df[LABEL_COL])
test_labels  = le.transform(test_df[LABEL_COL])

NUM_CLASSES = len(le.classes_)
print(f'Classes ({NUM_CLASSES}): {le.classes_.tolist()}')

label_map = {int(i): str(label) for i, label in enumerate(le.classes_)}
with open('label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)
print('Label map saved.')
print(label_map)

In [ ]:
# Cell 6: Tokenize
# Note: MAX_LENGTH=512 with BATCH_SIZE=8 fits in T4 VRAM
# Legal docs are long so 512 tokens captures more context than 256
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts, labels):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_tensors='pt'
    )
    return encodings, torch.tensor(labels, dtype=torch.long)

train_texts = train_df[TEXT_COL].astype(str).tolist()
val_texts   = val_df[TEXT_COL].astype(str).tolist()
test_texts  = test_df[TEXT_COL].astype(str).tolist()

print(f'Tokenizing {len(train_texts)} train samples (this takes ~3-4 mins)...')
train_enc, train_label_tensor = tokenize(train_texts, train_labels)
print(f'Tokenizing {len(val_texts)} val samples...')
val_enc,   val_label_tensor   = tokenize(val_texts,   val_labels)
print(f'Tokenizing {len(test_texts)} test samples...')
test_enc,  test_label_tensor  = tokenize(test_texts,  test_labels)
print('Tokenization complete.')

In [ ]:
# Cell 7: Dataset
class LegalDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

train_dataset = LegalDataset(train_enc, train_label_tensor)
val_dataset   = LegalDataset(val_enc,   val_label_tensor)
test_dataset  = LegalDataset(test_enc,  test_label_tensor)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

In [ ]:
# Cell 8: Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
    ignore_mismatched_sizes=True
)
model.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,}')
print(f'Num classes: {NUM_CLASSES}')

In [ ]:
# Cell 9: Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_weighted': f1_score(labels, predictions, average='weighted'),
        'f1_macro': f1_score(labels, predictions, average='macro')
    }

In [ ]:
# Cell 10: Training arguments
# gradient_accumulation_steps=4 simulates batch size of 32 without OOM
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=4,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to='none'
)

In [ ]:
# Cell 11: Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Starting training...')
print(f'Steps per epoch: {len(train_dataset) // (BATCH_SIZE * 4)}')
print(f'Total steps: {len(train_dataset) // (BATCH_SIZE * 4) * EPOCHS}')
train_result = trainer.train()
print('Training complete!')
print(f'Training loss: {train_result.training_loss:.4f}')

In [ ]:
# Cell 12: Evaluate on test set
print('Evaluating on test set...')
test_results    = trainer.predict(test_dataset)
test_preds      = np.argmax(test_results.predictions, axis=1)
test_labels_np  = test_label_tensor.numpy()

accuracy = accuracy_score(test_labels_np, test_preds)
f1_w     = f1_score(test_labels_np, test_preds, average='weighted')
f1_m     = f1_score(test_labels_np, test_preds, average='macro')

print(f'Test Accuracy:      {accuracy:.4f} ({accuracy*100:.2f}%)')
print(f'Test F1 (weighted): {f1_w:.4f}')
print(f'Test F1 (macro):    {f1_m:.4f}')
print('\nPer-class Report:')
print(classification_report(
    test_labels_np,
    test_preds,
    target_names=[str(c) for c in le.classes_]
))

In [ ]:
# Cell 13: Save model
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
shutil.copy('label_map.json', f'{SAVE_DIR}/label_map.json')

metrics = {
    'test_accuracy': round(float(accuracy), 4),
    'test_f1_weighted': round(float(f1_w), 4),
    'test_f1_macro': round(float(f1_m), 4),
    'model_name': MODEL_NAME,
    'num_classes': NUM_CLASSES,
    'classes': le.classes_.tolist(),
    'max_length': MAX_LENGTH,
    'train_samples': len(train_dataset),
    'dataset': 'LexGLUE 7-dataset combined'
}
with open(f'{SAVE_DIR}/training_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'Model saved to: {SAVE_DIR}')
print(f'Contents: {os.listdir(SAVE_DIR)}')

In [ ]:
# Cell 14: Zip and download
shutil.make_archive('legal_doc_classifier', 'zip', SAVE_DIR)
print('Zipped to: legal_doc_classifier.zip')

from google.colab import files
files.download('legal_doc_classifier.zip')
print('Download started!')

In [ ]:
# Cell 15: Inference test
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=SAVE_DIR,
    tokenizer=SAVE_DIR,
    device=0 if torch.cuda.is_available() else -1
)

with open(f'{SAVE_DIR}/label_map.json') as f:
    lm = {int(k): v for k, v in json.load(f).items()}

test_docs = [
    'The Court held that the Fourth Amendment prohibition on unreasonable searches applies to digital data stored on cell phones seized incident to arrest.',
    'The parties agree that this Agreement shall be governed by and construed in accordance with the laws of the State of Delaware.',
    'The Commission, having regard to the Treaty on the Functioning of the European Union, and in particular Article 108 thereof, hereby decides as follows.',
    'You agree that we may collect and use technical data and related information to facilitate software updates and product support.'
]

expected = ['scotus', 'ledgar', 'eurlex', 'unfair_tos']

print('Inference test:')
for doc, exp in zip(test_docs, expected):
    result = classifier(doc, truncation=True, max_length=MAX_LENGTH)
    label_id = int(result[0]['label'].replace('LABEL_', ''))
    predicted = lm.get(label_id, result[0]['label'])
    status = 'CORRECT' if predicted == exp else 'WRONG'
    print(f'  [{status}] Expected: {exp} | Predicted: {predicted} (score: {result[0]["score"]:.3f})')
    print(f'  Text: {doc[:80]}...')
    print()